# Personal Scheduler — Jupyter Notebook Version

This notebook contains the full **backend** of the Personal Scheduler project in one place:
constants → validators → models → SQLite database layer → services → demo → unit tests.

Layers (same architecture as the desktop app):

```
Scheduler (business logic) -> DatabaseManager (SQL) -> SQLite
ReminderService -> Scheduler
Models: ScheduleItem (abstract) -> Task, Event · Reminder · Category · User
```

**Note:** the PySide6 desktop GUI (windows, dialogs, Qt timer) cannot run inside a notebook, so it is replaced here by
a text demo and matplotlib charts. Run the cells from top to bottom. Requires Python 3.10+.

## 1. Imports

In [ ]:
import calendar
import sqlite3
import unittest
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import date, datetime, time, timedelta
from pathlib import Path

## 2. Constants

In [ ]:
PRIORITIES = ["Low", "Medium", "High"]
STATUSES = ["Pending", "In Progress", "Completed"]
RECURRENCES = ["None", "Daily", "Weekly", "Monthly"]

DEFAULT_CATEGORIES = [
    ("Study", "#4F8EF7"),
    ("Work", "#F59E0B"),
    ("Personal", "#10B981"),
    ("Exercise", "#EF4444"),
    ("Other", "#8B5CF6"),
]

PRIORITY_COLORS = {"Low": "#059669", "Medium": "#D97706", "High": "#DC2626"}

# (label shown to the user, minutes before the start time; None = no reminder)
REMINDER_OPTIONS = [
    ("No reminder", None),
    ("At start time", 0),
    ("5 minutes before", 5),
    ("10 minutes before", 10),
    ("15 minutes before", 15),
    ("30 minutes before", 30),
    ("1 hour before", 60),
    ("1 day before", 1440),
]

DATE_FORMAT = "%Y-%m-%d"
TIME_FORMAT = "%H:%M"
DATETIME_FORMAT = "%Y-%m-%d %H:%M"

## 3. Validators

In [ ]:
class Validator:
    """Groups all validation rules in one place.

    All methods are static/class methods: they need no object state, so we call
    them like Validator.title("..."). This keeps validation out of the GUI and
    the models (separation of responsibilities).
    """

    @staticmethod
    def title(value: str) -> str:
        value = (value or "").strip()
        if not value:
            raise ValueError("Title cannot be empty.")
        if len(value) > 100:
            raise ValueError("Title must be 100 characters or fewer.")
        return value

    @staticmethod
    def choice(value: str, allowed: list[str], field_name: str) -> str:
        if value not in allowed:
            raise ValueError(f"{field_name} must be one of: {', '.join(allowed)}.")
        return value

    @classmethod
    def priority(cls, value: str) -> str:
        return cls.choice(value, PRIORITIES, "Priority")

    @classmethod
    def status(cls, value: str) -> str:
        return cls.choice(value, STATUSES, "Status")

    @classmethod
    def recurrence(cls, value: str) -> str:
        return cls.choice(value, RECURRENCES, "Recurrence")

    @staticmethod
    def parse_date(text: str) -> date:
        try:
            return datetime.strptime(text, DATE_FORMAT).date()
        except (ValueError, TypeError):
            raise ValueError("Date must look like YYYY-MM-DD.")

    @staticmethod
    def parse_time(text: str) -> time:
        try:
            return datetime.strptime(text, TIME_FORMAT).time()
        except (ValueError, TypeError):
            raise ValueError("Time must look like HH:MM.")

    @staticmethod
    def parse_datetime(text: str) -> datetime:
        try:
            return datetime.strptime(text, DATETIME_FORMAT)
        except (ValueError, TypeError):
            raise ValueError("Date and time must look like YYYY-MM-DD HH:MM.")

    @staticmethod
    def time_range(start: time | None, end: time | None) -> None:
        """End time must be after start time (when both are given)."""
        if start and end and end <= start:
            raise ValueError("End time must be after start time.")

## 4. Models

`ScheduleItem` is abstract; `Task` and `Event` inherit from it (inheritance, abstraction, polymorphism).

In [ ]:
class ScheduleItem(ABC):
    """Common data and behavior shared by Task and Event.

    OOP concepts here:
    - ABSTRACTION: ABC + @abstractmethod means you cannot create a plain
      ScheduleItem; subclasses must implement get_type() and get_summary().
    - ENCAPSULATION: attributes start with an underscore and are exposed
      through properties, so setters can validate the data.
    - INHERITANCE: Task and Event reuse everything defined here.
    """

    def __init__(
        self,
        title: str,
        item_date: date,
        start_time: time | None = None,
        end_time: time | None = None,
        description: str = "",
        item_id: int | None = None,
        user_id: int | None = None,
    ) -> None:
        self._id = item_id
        self._user_id = user_id
        self.title = title            # goes through the validating setter
        self.description = description
        self._date = item_date
        Validator.time_range(start_time, end_time)
        self._start_time = start_time
        self._end_time = end_time

    # ----- properties (encapsulation) -----
    @property
    def id(self) -> int | None:
        return self._id

    @id.setter
    def id(self, value: int) -> None:
        # Only the database layer should assign this, after an INSERT.
        self._id = value

    @property
    def user_id(self) -> int | None:
        return self._user_id

    @user_id.setter
    def user_id(self, value: int | None) -> None:
        self._user_id = value

    @property
    def title(self) -> str:
        return self._title

    @title.setter
    def title(self, value: str) -> None:
        self._title = Validator.title(value)

    @property
    def description(self) -> str:
        return self._description

    @description.setter
    def description(self, value: str) -> None:
        self._description = (value or "").strip()

    @property
    def date(self) -> date:
        return self._date

    @date.setter
    def date(self, value: date) -> None:
        self._date = value

    @property
    def start_time(self) -> time | None:
        return self._start_time

    @property
    def end_time(self) -> time | None:
        return self._end_time

    def set_times(self, start: time | None, end: time | None) -> None:
        """Change both times together so the range is always validated."""
        Validator.time_range(start, end)
        self._start_time = start
        self._end_time = end

    # ----- abstract methods (polymorphism) -----
    @abstractmethod
    def get_type(self) -> str:
        """Return 'task' or 'event'."""

    @abstractmethod
    def get_summary(self) -> str:
        """Return a one-line text shown in lists."""

    # ----- shared behavior -----
    def starts_at(self) -> datetime:
        """Combine date and start time (midnight if no start time)."""
        return datetime.combine(self._date, self._start_time or time(0, 0))

    def overlaps_with(self, other: "ScheduleItem") -> bool:
        """True if both items are on the same day and their times overlap."""
        if self._date != other._date:
            return False
        if not (self._start_time and self._end_time
                and other._start_time and other._end_time):
            return False
        return self._start_time < other._end_time and other._start_time < self._end_time

    def time_text(self) -> str:
        """Readable time range such as '09:00 - 10:30'."""
        if self._start_time and self._end_time:
            return f"{self._start_time:%H:%M} - {self._end_time:%H:%M}"
        if self._start_time:
            return f"{self._start_time:%H:%M}"
        return "All day"

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(id={self._id}, title={self._title!r}, date={self._date})"

In [ ]:
class Task(ScheduleItem):
    """A normal task.

    INHERITANCE: Task extends ScheduleItem and adds priority, status,
    category and recurrence. super().__init__ reuses the parent constructor.
    """

    def __init__(
        self,
        title: str,
        item_date: date,
        start_time: time | None = None,
        end_time: time | None = None,
        description: str = "",
        priority: str = "Medium",
        status: str = "Pending",
        category_id: int | None = None,
        recurrence: str = "None",
        item_id: int | None = None,
        user_id: int | None = None,
    ) -> None:
        super().__init__(title, item_date, start_time, end_time,
                         description, item_id, user_id)
        self.priority = priority
        self.status = status
        self._category_id = category_id
        self.recurrence = recurrence

    @property
    def priority(self) -> str:
        return self._priority

    @priority.setter
    def priority(self, value: str) -> None:
        self._priority = Validator.priority(value)

    @property
    def status(self) -> str:
        return self._status

    @status.setter
    def status(self, value: str) -> None:
        self._status = Validator.status(value)

    @property
    def recurrence(self) -> str:
        return self._recurrence

    @recurrence.setter
    def recurrence(self, value: str) -> None:
        self._recurrence = Validator.recurrence(value)

    @property
    def category_id(self) -> int | None:
        return self._category_id

    @category_id.setter
    def category_id(self, value: int | None) -> None:
        self._category_id = value

    # ----- polymorphic methods: implement the abstract ones -----
    def get_type(self) -> str:
        return "task"

    def get_summary(self) -> str:
        return f"[{self._priority}] {self._title} ({self._status})"

    # ----- task-specific behavior -----
    def mark_completed(self) -> None:
        self._status = "Completed"

    def is_completed(self) -> bool:
        return self._status == "Completed"

    def is_overdue(self) -> bool:
        """A task is overdue if its time has passed and it isn't completed."""
        if self.is_completed():
            return False
        end = datetime.combine(self._date, self._end_time or time(23, 59))
        return end < datetime.now()

    def is_due_soon(self, hours: int = 24) -> bool:
        """True if the task starts within the next `hours` hours."""
        if self.is_completed():
            return False
        delta = self.starts_at() - datetime.now()
        return 0 <= delta.total_seconds() <= hours * 3600

In [ ]:
class Event(ScheduleItem):
    """An appointment or event.

    POLYMORPHISM: Event implements get_type() and get_summary() differently
    from Task. Code that loops over a list of ScheduleItem objects can call
    item.get_summary() without caring which subclass it is.
    """

    def __init__(
        self,
        title: str,
        item_date: date,
        start_time: time,
        end_time: time,
        description: str = "",
        location: str = "",
        item_id: int | None = None,
        user_id: int | None = None,
    ) -> None:
        super().__init__(title, item_date, start_time, end_time,
                         description, item_id, user_id)
        self._location = (location or "").strip()

    @property
    def location(self) -> str:
        return self._location

    @location.setter
    def location(self, value: str) -> None:
        self._location = (value or "").strip()

    def get_type(self) -> str:
        return "event"

    def get_summary(self) -> str:
        place = f" @ {self._location}" if self._location else ""
        return f"{self.time_text()}  {self._title}{place}"

    def duration_minutes(self) -> int:
        start = datetime.combine(self._date, self._start_time)
        end = datetime.combine(self._date, self._end_time)
        return int((end - start).total_seconds() // 60)

    def is_upcoming(self) -> bool:
        return self.starts_at() >= datetime.now()

In [ ]:
class Reminder:
    """Reminds the user about one ScheduleItem.

    A reminder stores only the item's type and id (not the item itself),
    which keeps the classes loosely coupled.
    """

    def __init__(
        self,
        item_type: str,
        item_id: int,
        remind_at: datetime,
        triggered: bool = False,
        reminder_id: int | None = None,
    ) -> None:
        if item_type not in ("task", "event"):
            raise ValueError("item_type must be 'task' or 'event'.")
        self._id = reminder_id
        self._item_type = item_type
        self._item_id = item_id
        self._remind_at = remind_at
        self._triggered = triggered

    @property
    def id(self) -> int | None:
        return self._id

    @id.setter
    def id(self, value: int) -> None:
        self._id = value

    @property
    def item_type(self) -> str:
        return self._item_type

    @property
    def item_id(self) -> int:
        return self._item_id

    @property
    def remind_at(self) -> datetime:
        return self._remind_at

    @property
    def triggered(self) -> bool:
        return self._triggered

    def is_due(self, now: datetime | None = None) -> bool:
        """True when the time has arrived and we haven't notified yet."""
        now = now or datetime.now()
        return not self._triggered and self._remind_at <= now

    def mark_triggered(self) -> None:
        self._triggered = True

    def __repr__(self) -> str:
        return f"Reminder({self._item_type} #{self._item_id} at {self._remind_at:%Y-%m-%d %H:%M})"

In [ ]:
class Category:
    """A task category such as Study or Work."""

    def __init__(self, name: str, color: str = "#8B5CF6",
                 category_id: int | None = None) -> None:
        name = (name or "").strip()
        if not name:
            raise ValueError("Category name cannot be empty.")
        self._id = category_id
        self._name = name
        self._color = color

    @property
    def id(self) -> int | None:
        return self._id

    @property
    def name(self) -> str:
        return self._name

    @property
    def color(self) -> str:
        return self._color

    def __str__(self) -> str:
        return self._name

    def __repr__(self) -> str:
        return f"Category(id={self._id}, name={self._name!r})"

In [ ]:
class User:
    """Stores the user's name and email."""

    def __init__(self, name: str, email: str = "",
                 user_id: int | None = None,
                 created_at: datetime | None = None) -> None:
        name = (name or "").strip()
        if not name:
            raise ValueError("Name cannot be empty.")
        self._id = user_id
        self._name = name
        self._email = (email or "").strip()
        self._created_at = created_at or datetime.now()

    @property
    def id(self) -> int | None:
        return self._id

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        value = (value or "").strip()
        if not value:
            raise ValueError("Name cannot be empty.")
        self._name = value

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, value: str) -> None:
        self._email = (value or "").strip()

    @property
    def created_at(self) -> datetime:
        return self._created_at

    def greeting(self) -> str:
        hour = datetime.now().hour
        part = "morning" if hour < 12 else "afternoon" if hour < 18 else "evening"
        return f"Good {part}, {self._name}!"

## 5. Database layer

The only place that contains SQL. In the notebook the default file is `data/scheduler.db` next to the notebook; use `":memory:"` for a throw-away database.

In [ ]:
class DatabaseError(Exception):
    """Raised when a database operation fails.

    We wrap sqlite3 errors in our own exception so the rest of the app
    does not need to know that SQLite is being used (abstraction).
    """


class DatabaseManager:
    """Creates the database and performs every read/write operation.

    ENCAPSULATION: the connection is private (_conn). Other classes can only
    use the public methods below.
    """

    # Default location: <project folder>/data/scheduler.db
    DEFAULT_PATH = Path.cwd() / "data" / "scheduler.db"

    def __init__(self, db_path: str | Path | None = None) -> None:
        self._db_path = str(db_path) if db_path else str(self.DEFAULT_PATH)

        if self._db_path != ":memory:":
            Path(self._db_path).parent.mkdir(parents=True, exist_ok=True)

        try:
            self._conn = sqlite3.connect(self._db_path)
        except sqlite3.Error as error:
            raise DatabaseError(f"Could not open database: {error}") from error

        self._conn.row_factory = sqlite3.Row          # access columns by name
        self._conn.execute("PRAGMA foreign_keys = ON")

        self.create_tables()
        self._seed_defaults()

    # ------------------------------------------------------------------
    # Low-level helpers (private)
    # ------------------------------------------------------------------
    def _execute(self, sql: str, params: tuple = ()) -> sqlite3.Cursor:
        """Run INSERT/UPDATE/DELETE. Always uses ? placeholders (parameterized)."""
        try:
            with self._conn:                          # commits, or rolls back on error
                return self._conn.execute(sql, params)
        except sqlite3.Error as error:
            raise DatabaseError(f"Database error: {error}") from error

    def _query(self, sql: str, params: tuple = ()) -> list[sqlite3.Row]:
        """Run a SELECT and return all rows."""
        try:
            return self._conn.execute(sql, params).fetchall()
        except sqlite3.Error as error:
            raise DatabaseError(f"Database error: {error}") from error

    @staticmethod
    def _time_to_text(value: time | None) -> str | None:
        return value.strftime(TIME_FORMAT) if value else None

    @staticmethod
    def _text_to_time(text: str | None) -> time | None:
        return datetime.strptime(text, TIME_FORMAT).time() if text else None

    @staticmethod
    def _text_to_date(text: str) -> date:
        return datetime.strptime(text, DATE_FORMAT).date()

    # ------------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------------
    def create_tables(self) -> None:
        """Create all tables if they do not exist yet."""
        try:
            self._conn.executescript(
                """
                CREATE TABLE IF NOT EXISTS users (
                    id          INTEGER PRIMARY KEY AUTOINCREMENT,
                    name        TEXT NOT NULL,
                    email       TEXT,
                    created_at  TEXT NOT NULL
                );

                CREATE TABLE IF NOT EXISTS categories (
                    id      INTEGER PRIMARY KEY AUTOINCREMENT,
                    name    TEXT NOT NULL UNIQUE,
                    color   TEXT NOT NULL
                );

                CREATE TABLE IF NOT EXISTS tasks (
                    id           INTEGER PRIMARY KEY AUTOINCREMENT,
                    title        TEXT NOT NULL,
                    description  TEXT,
                    date         TEXT NOT NULL,
                    start_time   TEXT,
                    end_time     TEXT,
                    priority     TEXT NOT NULL,
                    status       TEXT NOT NULL,
                    category_id  INTEGER REFERENCES categories(id) ON DELETE SET NULL,
                    recurrence   TEXT NOT NULL DEFAULT 'None',
                    user_id      INTEGER REFERENCES users(id) ON DELETE SET NULL
                );

                CREATE TABLE IF NOT EXISTS events (
                    id           INTEGER PRIMARY KEY AUTOINCREMENT,
                    title        TEXT NOT NULL,
                    description  TEXT,
                    date         TEXT NOT NULL,
                    start_time   TEXT NOT NULL,
                    end_time     TEXT NOT NULL,
                    location     TEXT,
                    user_id      INTEGER REFERENCES users(id) ON DELETE SET NULL
                );

                CREATE TABLE IF NOT EXISTS reminders (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    item_type  TEXT NOT NULL,
                    item_id    INTEGER NOT NULL,
                    remind_at  TEXT NOT NULL,
                    triggered  INTEGER NOT NULL DEFAULT 0
                );
                """
            )
        except sqlite3.Error as error:
            raise DatabaseError(f"Could not create tables: {error}") from error

    def _seed_defaults(self) -> None:
        """Insert the default categories and a default user on first run."""
        if not self._query("SELECT id FROM categories LIMIT 1"):
            for name, color in DEFAULT_CATEGORIES:
                self._execute(
                    "INSERT INTO categories (name, color) VALUES (?, ?)",
                    (name, color),
                )
        if not self._query("SELECT id FROM users LIMIT 1"):
            self._execute(
                "INSERT INTO users (name, email, created_at) VALUES (?, ?, ?)",
                ("Student", "", datetime.now().strftime(DATETIME_FORMAT)),
            )

    def close(self) -> None:
        """Close the connection (call this when the app exits)."""
        self._conn.close()

    # ------------------------------------------------------------------
    # Users
    # ------------------------------------------------------------------
    def get_user(self) -> User:
        """Return the (single) user of this app."""
        row = self._query("SELECT * FROM users ORDER BY id LIMIT 1")[0]
        return User(
            name=row["name"],
            email=row["email"] or "",
            user_id=row["id"],
            created_at=datetime.strptime(row["created_at"], DATETIME_FORMAT),
        )

    def update_user(self, user: User) -> None:
        self._execute(
            "UPDATE users SET name = ?, email = ? WHERE id = ?",
            (user.name, user.email, user.id),
        )

    # ------------------------------------------------------------------
    # Categories
    # ------------------------------------------------------------------
    def get_categories(self) -> list[Category]:
        rows = self._query("SELECT * FROM categories ORDER BY id")
        return [Category(r["name"], r["color"], r["id"]) for r in rows]

    def add_category(self, category: Category) -> int:
        cursor = self._execute(
            "INSERT INTO categories (name, color) VALUES (?, ?)",
            (category.name, category.color),
        )
        return cursor.lastrowid

    # ------------------------------------------------------------------
    # Tasks
    # ------------------------------------------------------------------
    def _row_to_task(self, row: sqlite3.Row) -> Task:
        """Convert one database row into a Task object."""
        return Task(
            title=row["title"],
            item_date=self._text_to_date(row["date"]),
            start_time=self._text_to_time(row["start_time"]),
            end_time=self._text_to_time(row["end_time"]),
            description=row["description"] or "",
            priority=row["priority"],
            status=row["status"],
            category_id=row["category_id"],
            recurrence=row["recurrence"],
            item_id=row["id"],
            user_id=row["user_id"],
        )

    def insert_task(self, task: Task) -> int:
        """Save a new task; the new id is stored on the object and returned."""
        cursor = self._execute(
            """INSERT INTO tasks (title, description, date, start_time, end_time,
                                  priority, status, category_id, recurrence, user_id)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (
                task.title, task.description, task.date.strftime(DATE_FORMAT),
                self._time_to_text(task.start_time), self._time_to_text(task.end_time),
                task.priority, task.status, task.category_id,
                task.recurrence, task.user_id,
            ),
        )
        task.id = cursor.lastrowid
        return task.id

    def update_task(self, task: Task) -> None:
        self._execute(
            """UPDATE tasks SET title = ?, description = ?, date = ?, start_time = ?,
                                end_time = ?, priority = ?, status = ?,
                                category_id = ?, recurrence = ?
               WHERE id = ?""",
            (
                task.title, task.description, task.date.strftime(DATE_FORMAT),
                self._time_to_text(task.start_time), self._time_to_text(task.end_time),
                task.priority, task.status, task.category_id,
                task.recurrence, task.id,
            ),
        )

    def delete_task(self, task_id: int) -> None:
        self.delete_reminders_for_item("task", task_id)
        self._execute("DELETE FROM tasks WHERE id = ?", (task_id,))

    def get_task(self, task_id: int) -> Task | None:
        rows = self._query("SELECT * FROM tasks WHERE id = ?", (task_id,))
        return self._row_to_task(rows[0]) if rows else None

    def get_tasks(
        self,
        item_date: date | None = None,
        category_id: int | None = None,
        priority: str | None = None,
        status: str | None = None,
        keyword: str | None = None,
    ) -> list[Task]:
        """Return tasks, optionally filtered. Any argument left as None is ignored.

        This one method covers both FILTERING and SEARCH. The column names in
        the SQL are fixed text we wrote ourselves; only the VALUES come from
        the user, and those always go through ? placeholders.
        """
        conditions: list[str] = []
        params: list = []

        if item_date is not None:
            conditions.append("date = ?")
            params.append(item_date.strftime(DATE_FORMAT))
        if category_id is not None:
            conditions.append("category_id = ?")
            params.append(category_id)
        if priority:
            conditions.append("priority = ?")
            params.append(priority)
        if status:
            conditions.append("status = ?")
            params.append(status)
        if keyword:
            conditions.append("(title LIKE ? OR description LIKE ?)")
            params.extend([f"%{keyword}%", f"%{keyword}%"])

        sql = "SELECT * FROM tasks"
        if conditions:
            sql += " WHERE " + " AND ".join(conditions)
        sql += " ORDER BY date, start_time"

        return [self._row_to_task(r) for r in self._query(sql, tuple(params))]

    # ------------------------------------------------------------------
    # Events
    # ------------------------------------------------------------------
    def _row_to_event(self, row: sqlite3.Row) -> Event:
        return Event(
            title=row["title"],
            item_date=self._text_to_date(row["date"]),
            start_time=self._text_to_time(row["start_time"]),
            end_time=self._text_to_time(row["end_time"]),
            description=row["description"] or "",
            location=row["location"] or "",
            item_id=row["id"],
            user_id=row["user_id"],
        )

    def insert_event(self, event: Event) -> int:
        cursor = self._execute(
            """INSERT INTO events (title, description, date, start_time, end_time,
                                   location, user_id)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (
                event.title, event.description, event.date.strftime(DATE_FORMAT),
                self._time_to_text(event.start_time), self._time_to_text(event.end_time),
                event.location, event.user_id,
            ),
        )
        event.id = cursor.lastrowid
        return event.id

    def update_event(self, event: Event) -> None:
        self._execute(
            """UPDATE events SET title = ?, description = ?, date = ?,
                                 start_time = ?, end_time = ?, location = ?
               WHERE id = ?""",
            (
                event.title, event.description, event.date.strftime(DATE_FORMAT),
                self._time_to_text(event.start_time), self._time_to_text(event.end_time),
                event.location, event.id,
            ),
        )

    def delete_event(self, event_id: int) -> None:
        self.delete_reminders_for_item("event", event_id)
        self._execute("DELETE FROM events WHERE id = ?", (event_id,))

    def get_event(self, event_id: int) -> Event | None:
        rows = self._query("SELECT * FROM events WHERE id = ?", (event_id,))
        return self._row_to_event(rows[0]) if rows else None

    def get_events(
        self,
        item_date: date | None = None,
        from_date: date | None = None,
        keyword: str | None = None,
    ) -> list[Event]:
        """Return events. `from_date` returns that day and everything after it."""
        conditions: list[str] = []
        params: list = []

        if item_date is not None:
            conditions.append("date = ?")
            params.append(item_date.strftime(DATE_FORMAT))
        if from_date is not None:
            conditions.append("date >= ?")
            params.append(from_date.strftime(DATE_FORMAT))
        if keyword:
            conditions.append("(title LIKE ? OR description LIKE ? OR location LIKE ?)")
            params.extend([f"%{keyword}%"] * 3)

        sql = "SELECT * FROM events"
        if conditions:
            sql += " WHERE " + " AND ".join(conditions)
        sql += " ORDER BY date, start_time"

        return [self._row_to_event(r) for r in self._query(sql, tuple(params))]

    # ------------------------------------------------------------------
    # Reminders
    # ------------------------------------------------------------------
    def _row_to_reminder(self, row: sqlite3.Row) -> Reminder:
        return Reminder(
            item_type=row["item_type"],
            item_id=row["item_id"],
            remind_at=datetime.strptime(row["remind_at"], DATETIME_FORMAT),
            triggered=bool(row["triggered"]),
            reminder_id=row["id"],
        )

    def insert_reminder(self, reminder: Reminder) -> int:
        cursor = self._execute(
            "INSERT INTO reminders (item_type, item_id, remind_at, triggered) "
            "VALUES (?, ?, ?, ?)",
            (
                reminder.item_type, reminder.item_id,
                reminder.remind_at.strftime(DATETIME_FORMAT), int(reminder.triggered),
            ),
        )
        reminder.id = cursor.lastrowid
        return reminder.id

    def get_reminders_for_item(self, item_type: str, item_id: int) -> list[Reminder]:
        rows = self._query(
            "SELECT * FROM reminders WHERE item_type = ? AND item_id = ?",
            (item_type, item_id),
        )
        return [self._row_to_reminder(r) for r in rows]

    def get_due_reminders(self, now: datetime | None = None) -> list[Reminder]:
        """Reminders whose time has arrived and that were not shown yet."""
        now = now or datetime.now()
        rows = self._query(
            "SELECT * FROM reminders WHERE triggered = 0 AND remind_at <= ? "
            "ORDER BY remind_at",
            (now.strftime(DATETIME_FORMAT),),
        )
        return [self._row_to_reminder(r) for r in rows]

    def mark_reminder_triggered(self, reminder_id: int) -> None:
        self._execute("UPDATE reminders SET triggered = 1 WHERE id = ?", (reminder_id,))

    def delete_reminders_for_item(self, item_type: str, item_id: int) -> None:
        self._execute(
            "DELETE FROM reminders WHERE item_type = ? AND item_id = ?",
            (item_type, item_id),
        )

## 6. Services

Business rules: filtering, search, recurring tasks, conflicts, statistics, reminders.

In [ ]:
class Scheduler:
    """Manages tasks, events, reminders and statistics.

    COMPOSITION: a Scheduler HAS a DatabaseManager and a User, and it manages
    collections of Task and Event objects (both are ScheduleItems).
    FACADE: the GUI only needs this one class instead of many.
    """

    def __init__(self, db: DatabaseManager) -> None:
        self._db = db
        self._user: User = db.get_user()
        self._categories: dict[int, Category] = {c.id: c for c in db.get_categories()}

    # ------------------------------------------------------------------
    # User and categories
    # ------------------------------------------------------------------
    @property
    def user(self) -> User:
        return self._user

    def update_user(self, name: str, email: str = "") -> None:
        self._user.name = name          # the setter validates
        self._user.email = email
        self._db.update_user(self._user)

    def get_categories(self) -> list[Category]:
        return list(self._categories.values())

    def add_category(self, category: Category) -> None:
        category_id = self._db.add_category(category)
        self._categories[category_id] = Category(category.name, category.color, category_id)

    def get_category_name(self, category_id: int | None) -> str:
        category = self._categories.get(category_id)
        return category.name if category else "Uncategorized"

    # ------------------------------------------------------------------
    # Tasks
    # ------------------------------------------------------------------
    def add_task(self, task: Task, reminder_minutes: int | None = None) -> Task:
        self._validate_reminder(task, reminder_minutes)
        task.user_id = self._user.id
        self._db.insert_task(task)
        self._save_reminder(task, reminder_minutes)
        return task

    def update_task(self, task: Task, reminder_minutes: int | None = None) -> Task | None:
        """Save changes. Returns the next occurrence if a recurring task was completed."""
        self._validate_reminder(task, reminder_minutes)
        previous = self._db.get_task(task.id)
        was_completed = previous.is_completed() if previous else False
        self._db.update_task(task)
        self._save_reminder(task, reminder_minutes)
        return self._after_status_change(task, was_completed)

    def delete_task(self, task_id: int) -> None:
        self._db.delete_task(task_id)

    def get_task(self, task_id: int) -> Task | None:
        return self._db.get_task(task_id)

    def get_tasks(self, **filters) -> list[Task]:
        """Filter by item_date, category_id, priority, status and/or keyword."""
        return self._db.get_tasks(**filters)

    def search_tasks(self, keyword: str) -> list[Task]:
        return self._db.get_tasks(keyword=keyword)

    def set_task_status(self, task_id: int, status: str) -> Task | None:
        """Change only the status (Pending / In Progress / Completed)."""
        task = self._db.get_task(task_id)
        if task is None:
            raise ValueError("Task not found.")
        was_completed = task.is_completed()
        task.status = status
        self._db.update_task(task)
        return self._after_status_change(task, was_completed)

    # ------------------------------------------------------------------
    # Events
    # ------------------------------------------------------------------
    def add_event(self, event: Event, reminder_minutes: int | None = None) -> Event:
        self._validate_reminder(event, reminder_minutes)
        event.user_id = self._user.id
        self._db.insert_event(event)
        self._save_reminder(event, reminder_minutes)
        return event

    def update_event(self, event: Event, reminder_minutes: int | None = None) -> None:
        self._validate_reminder(event, reminder_minutes)
        self._db.update_event(event)
        self._save_reminder(event, reminder_minutes)

    def delete_event(self, event_id: int) -> None:
        self._db.delete_event(event_id)

    def get_event(self, event_id: int) -> Event | None:
        return self._db.get_event(event_id)

    def get_events(self, **filters) -> list[Event]:
        """Filter by item_date, from_date and/or keyword."""
        return self._db.get_events(**filters)

    def find_conflicts(self, event: Event) -> list[Event]:
        """Other events on the same day whose time overlaps with `event`."""
        same_day = self._db.get_events(item_date=event.date)
        return [e for e in same_day if e.id != event.id and event.overlaps_with(e)]

    # ------------------------------------------------------------------
    # Working with both kinds of item (polymorphism)
    # ------------------------------------------------------------------
    def get_item(self, item_type: str, item_id: int) -> ScheduleItem | None:
        if item_type == "task":
            return self._db.get_task(item_id)
        return self._db.get_event(item_id)

    def items_on(self, day: date) -> list[ScheduleItem]:
        """Tasks AND events of one day, sorted by start time."""
        items: list[ScheduleItem] = []
        items.extend(self._db.get_tasks(item_date=day))
        items.extend(self._db.get_events(item_date=day))
        return sorted(items, key=lambda item: item.starts_at())

    def dates_with_items(self, year: int, month: int) -> set[date]:
        """Days of a month that have at least one task or event."""
        all_items: list[ScheduleItem] = [*self._db.get_tasks(), *self._db.get_events()]
        return {i.date for i in all_items if i.date.year == year and i.date.month == month}

    # ------------------------------------------------------------------
    # Dashboard data
    # ------------------------------------------------------------------
    def todays_tasks(self) -> list[Task]:
        return self._db.get_tasks(item_date=date.today())

    def upcoming_events(self, limit: int = 5) -> list[Event]:
        events = self._db.get_events(from_date=date.today())
        return [e for e in events if e.is_upcoming()][:limit]

    def tasks_due_soon(self, hours: int = 24) -> list[Task]:
        return [t for t in self._db.get_tasks() if t.is_due_soon(hours)]

    def get_statistics(self) -> dict:
        tasks = self._db.get_tasks()
        total = len(tasks)
        completed = sum(1 for t in tasks if t.status == "Completed")
        pending = sum(1 for t in tasks if t.status == "Pending")
        in_progress = sum(1 for t in tasks if t.status == "In Progress")

        by_category: dict[str, int] = {}
        for task in tasks:
            name = self.get_category_name(task.category_id)
            by_category[name] = by_category.get(name, 0) + 1

        return {
            "total": total,
            "completed": completed,
            "pending": pending,
            "in_progress": in_progress,
            "unfinished": pending + in_progress,
            "high_priority": sum(1 for t in tasks if t.priority == "High"),
            "overdue": sum(1 for t in tasks if t.is_overdue()),
            "events": len(self._db.get_events()),
            "completion_percentage": round(completed / total * 100, 1) if total else 0.0,
            "by_category": by_category,
        }

    # ------------------------------------------------------------------
    # Reminders (thin delegation used by ReminderService)
    # ------------------------------------------------------------------
    def get_due_reminders(self, now: datetime | None = None) -> list[Reminder]:
        return self._db.get_due_reminders(now)

    def mark_reminder_triggered(self, reminder: Reminder) -> None:
        reminder.mark_triggered()
        self._db.mark_reminder_triggered(reminder.id)

    def get_reminder_minutes(self, item: ScheduleItem) -> int | None:
        """How many minutes before the start the item's reminder is set (None = no reminder)."""
        reminders = self._db.get_reminders_for_item(item.get_type(), item.id)
        if not reminders:
            return None
        delta = item.starts_at() - reminders[0].remind_at
        return max(0, int(delta.total_seconds() // 60))

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _validate_reminder(item: ScheduleItem, minutes: int | None) -> None:
        if minutes is not None and item.start_time is None:
            raise ValueError("A reminder needs a start time.")

    def _save_reminder(self, item: ScheduleItem, minutes: int | None) -> None:
        """Replace the item's reminder (if any) with a new one."""
        self._db.delete_reminders_for_item(item.get_type(), item.id)
        if minutes is None:
            return
        remind_at = item.starts_at() - timedelta(minutes=minutes)
        already_past = remind_at < datetime.now()   # never fire reminders in the past
        self._db.insert_reminder(
            Reminder(item.get_type(), item.id, remind_at, triggered=already_past)
        )

    def _after_status_change(self, task: Task, was_completed: bool) -> Task | None:
        """When a recurring task becomes Completed, create its next occurrence."""
        if task.is_completed() and not was_completed and task.recurrence != "None":
            return self._create_next_occurrence(task)
        return None

    def _create_next_occurrence(self, task: Task) -> Task | None:
        next_date = self._next_date(task.date, task.recurrence)
        already_exists = any(t.title == task.title for t in self._db.get_tasks(item_date=next_date))
        if already_exists:
            return None
        minutes = self.get_reminder_minutes(task)
        new_task = Task(
            title=task.title, item_date=next_date,
            start_time=task.start_time, end_time=task.end_time,
            description=task.description, priority=task.priority,
            status="Pending", category_id=task.category_id,
            recurrence=task.recurrence, user_id=task.user_id,
        )
        self._db.insert_task(new_task)
        self._save_reminder(new_task, minutes)
        return new_task

    @staticmethod
    def _next_date(current: date, recurrence: str) -> date:
        if recurrence == "Daily":
            return current + timedelta(days=1)
        if recurrence == "Weekly":
            return current + timedelta(weeks=1)
        # Monthly: keep the day number, but never go past the end of the month
        month_index = current.month  # next month, 0-based arithmetic below
        year = current.year + month_index // 12
        month = month_index % 12 + 1
        day = min(current.day, calendar.monthrange(year, month)[1])
        return date(year, month, day)

In [ ]:
@dataclass
class Notification:
    """A message the GUI can show to the user."""
    title: str
    message: str
    item_type: str


class ReminderService:
    """Checks for due reminders.

    This class has NO GUI code. The GUI calls check_due() on a timer and decides
    how to display the returned notifications (separation of responsibilities).
    """

    STALE_AFTER = timedelta(hours=24)   # reminders older than this are skipped silently

    def __init__(self, scheduler: Scheduler) -> None:
        self._scheduler = scheduler

    def check_due(self, now: datetime | None = None) -> list[Notification]:
        now = now or datetime.now()
        notifications: list[Notification] = []

        for reminder in self._scheduler.get_due_reminders(now):
            self._scheduler.mark_reminder_triggered(reminder)   # never show twice

            if now - reminder.remind_at > self.STALE_AFTER:
                continue
            item = self._scheduler.get_item(reminder.item_type, reminder.item_id)
            if item is None:
                continue
            if isinstance(item, Task) and item.is_completed():
                continue

            kind = "Task" if reminder.item_type == "task" else "Event"
            notifications.append(Notification(
                title=f"{kind} reminder",
                message=f"{item.get_summary()}\nStarts {item.starts_at():%A %d %B at %H:%M}",
                item_type=reminder.item_type,
            ))
        return notifications

## 7. Demo

We use an in-memory database so nothing is written to disk.

In [ ]:
db = DatabaseManager(":memory:")
scheduler = Scheduler(db)
reminders = ReminderService(scheduler)

print(scheduler.user.greeting())
print("Categories:", [c.name for c in scheduler.get_categories()])

In [ ]:
today = date.today()
study = {c.name: c.id for c in scheduler.get_categories()}

scheduler.add_task(Task("Study OOP chapter 5", today, time(9), time(10), "Inheritance + polymorphism",
                        priority="High", category_id=study["Study"]), reminder_minutes=15)
scheduler.add_task(Task("Go for a run", today, time(18), time(19), priority="Low",
                        category_id=study["Exercise"], recurrence="Daily"))
scheduler.add_task(Task("Submit assignment", today - timedelta(days=1), priority="High",
                        category_id=study["Study"]))
scheduler.add_event(Event("Team meeting", today, time(10), time(11), location="Room 5"))

for item in scheduler.items_on(today):          # polymorphism: same call, different output
    print(f"{item.get_type():5} | {item.get_summary()}")

In [ ]:
# Overlap detection
clash = Event("Doctor", today, time(10, 30), time(11, 30))
print("Conflicts:", [e.title for e in scheduler.find_conflicts(clash)])

# Search and filters
print("Search 'run':", [t.title for t in scheduler.search_tasks("run")])
print("High priority:", [t.title for t in scheduler.get_tasks(priority="High")])

In [ ]:
# Recurring task: completing it creates the next occurrence
run = scheduler.get_tasks(keyword="run")[0]
next_run = scheduler.set_task_status(run.id, "Completed")
print("Next occurrence:", next_run)

In [ ]:
# Reminders: pretend the clock is 10 minutes before the study task
service_time = datetime.combine(today, time(8, 50))
for n in reminders.check_due(service_time):
    print(n.title, "->", n.message)

## 8. Statistics

In [ ]:
stats = scheduler.get_statistics()
stats

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(["Pending", "In Progress", "Completed"],
            [stats["pending"], stats["in_progress"], stats["completed"]],
            color=["#D97706", "#4F8EF7", "#059669"])
axes[0].set_title(f"Task status ({stats['completion_percentage']}% done)")

if stats["by_category"]:
    axes[1].pie(stats["by_category"].values(), labels=stats["by_category"].keys(), autopct="%d")
axes[1].set_title("Tasks by category")

plt.tight_layout()
plt.show()

## 9. Unit tests

The original tests for models, validators, database, scheduler rules and reminders (the GUI smoke test is left out).

In [ ]:
class ModelTests(unittest.TestCase):
    def test_schedule_item_is_abstract(self):
        with self.assertRaises(TypeError):
            ScheduleItem("x", date.today())          # abstract: cannot be created

    def test_polymorphism(self):
        items = [
            Task("Study", date.today(), time(9), time(10), priority="High"),
            Event("Meeting", date.today(), time(10), time(11), location="Room 5"),
        ]
        self.assertEqual([i.get_type() for i in items], ["task", "event"])
        self.assertIn("[High]", items[0].get_summary())
        self.assertIn("Room 5", items[1].get_summary())

    def test_validation_in_setters(self):
        task = Task("Study", date.today())
        with self.assertRaises(ValueError):
            task.priority = "Urgent"
        with self.assertRaises(ValueError):
            task.title = "   "
        with self.assertRaises(ValueError):
            Task("Bad times", date.today(), time(10), time(9))

    def test_overlap(self):
        a = Event("A", date.today(), time(9), time(10))
        b = Event("B", date.today(), time(9, 30), time(11))
        c = Event("C", date.today(), time(10), time(11))
        self.assertTrue(a.overlaps_with(b))
        self.assertFalse(a.overlaps_with(c))        # touching is not overlapping

    def test_overdue_and_completed(self):
        old = Task("Old", date.today() - timedelta(days=2))
        self.assertTrue(old.is_overdue())
        old.mark_completed()
        self.assertFalse(old.is_overdue())


class ValidatorTests(unittest.TestCase):
    def test_parsing(self):
        self.assertEqual(Validator.parse_date("2026-10-02"), date(2026, 10, 2))
        self.assertEqual(Validator.parse_time("09:30"), time(9, 30))
        with self.assertRaises(ValueError):
            Validator.parse_date("02/10/2026")


class SchedulerTestCase(unittest.TestCase):
    def setUp(self):
        self.db = DatabaseManager(":memory:")
        self.scheduler = Scheduler(self.db)

    def tearDown(self):
        self.db.close()


class DatabaseTests(SchedulerTestCase):
    def test_defaults_created(self):
        names = [c.name for c in self.db.get_categories()]
        self.assertEqual(names, ["Study", "Work", "Personal", "Exercise", "Other"])
        self.assertEqual(self.db.get_user().name, "Student")

    def test_task_crud(self):
        task = Task("Study OOP", date.today(), time(9), time(10), "Chapter 5", priority="High")
        self.db.insert_task(task)
        self.assertIsNotNone(task.id)
        task.status = "In Progress"
        self.db.update_task(task)
        self.assertEqual(self.db.get_task(task.id).status, "In Progress")
        self.db.delete_task(task.id)
        self.assertIsNone(self.db.get_task(task.id))

    def test_sql_injection_is_harmless(self):
        self.db.insert_task(Task("Normal", date.today()))
        self.assertEqual(self.db.get_tasks(keyword="'; DROP TABLE tasks; --"), [])
        self.assertEqual(len(self.db.get_tasks()), 1)


class SchedulerTests(SchedulerTestCase):
    def test_search_and_filters(self):
        today = date.today()
        self.scheduler.add_task(Task("Write report", today, priority="High", description="for work"))
        self.scheduler.add_task(Task("Go running", today, priority="Low", status="Completed"))
        self.assertEqual(len(self.scheduler.search_tasks("report")), 1)
        self.assertEqual(len(self.scheduler.search_tasks("work")), 1)      # description
        self.assertEqual(len(self.scheduler.get_tasks(priority="Low")), 1)
        self.assertEqual(len(self.scheduler.get_tasks(status="Completed", item_date=today)), 1)

    def test_statistics(self):
        today = date.today()
        self.scheduler.add_task(Task("A", today, priority="High", status="Completed"))
        self.scheduler.add_task(Task("B", today, status="Pending"))
        self.scheduler.add_task(Task("C", today, status="In Progress"))
        self.scheduler.add_task(Task("D", today, status="Pending"))
        stats = self.scheduler.get_statistics()
        self.assertEqual(stats["total"], 4)
        self.assertEqual(stats["completed"], 1)
        self.assertEqual(stats["unfinished"], 3)
        self.assertEqual(stats["high_priority"], 1)
        self.assertEqual(stats["completion_percentage"], 25.0)

    def test_empty_statistics(self):
        self.assertEqual(self.scheduler.get_statistics()["completion_percentage"], 0.0)

    def test_recurring_daily_and_weekly(self):
        daily = self.scheduler.add_task(Task("Daily", date(2026, 10, 2), recurrence="Daily"))
        new = self.scheduler.set_task_status(daily.id, "Completed")
        self.assertEqual(new.date, date(2026, 10, 3))
        self.assertEqual(new.status, "Pending")
        # completing again must not create a duplicate
        self.scheduler.set_task_status(daily.id, "Pending")
        self.assertIsNone(self.scheduler.set_task_status(daily.id, "Completed"))

        weekly = self.scheduler.add_task(Task("Weekly", date(2026, 10, 2), recurrence="Weekly"))
        self.assertEqual(self.scheduler.set_task_status(weekly.id, "Completed").date, date(2026, 10, 9))

    def test_recurring_monthly_end_of_month(self):
        task = self.scheduler.add_task(Task("Rent", date(2026, 1, 31), recurrence="Monthly"))
        self.assertEqual(self.scheduler.set_task_status(task.id, "Completed").date, date(2026, 2, 28))
        dec = self.scheduler.add_task(Task("Year end", date(2026, 12, 15), recurrence="Monthly"))
        self.assertEqual(self.scheduler.set_task_status(dec.id, "Completed").date, date(2027, 1, 15))

    def test_event_conflicts(self):
        today = date.today()
        first = self.scheduler.add_event(Event("A", today, time(9), time(10)))
        clash = Event("B", today, time(9, 30), time(10, 30))
        self.assertEqual([e.id for e in self.scheduler.find_conflicts(clash)], [first.id])
        self.assertEqual(self.scheduler.find_conflicts(first), [])           # not with itself

    def test_items_on_mixes_tasks_and_events_sorted(self):
        today = date.today()
        self.scheduler.add_event(Event("Late event", today, time(15), time(16)))
        self.scheduler.add_task(Task("Early task", today, time(8), time(9)))
        types = [i.get_type() for i in self.scheduler.items_on(today)]
        self.assertEqual(types, ["task", "event"])

    def test_reminder_needs_start_time(self):
        with self.assertRaises(ValueError):
            self.scheduler.add_task(Task("No time", date.today()), reminder_minutes=10)

    def test_delete_removes_reminders(self):
        later = datetime.now() + timedelta(days=2)
        task = self.scheduler.add_task(
            Task("T", later.date(), later.time().replace(second=0, microsecond=0)), reminder_minutes=15)
        self.assertEqual(self.scheduler.get_reminder_minutes(task), 15)
        self.scheduler.delete_task(task.id)
        self.assertEqual(self.db.get_due_reminders(datetime.now() + timedelta(days=30)), [])


class ReminderServiceTests(SchedulerTestCase):
    def test_due_reminder_fires_once(self):
        start = datetime.now() + timedelta(hours=2)
        task = self.scheduler.add_task(
            Task("Call mom", start.date(), start.time().replace(second=0, microsecond=0)),
            reminder_minutes=30)
        service = ReminderService(self.scheduler)

        self.assertEqual(service.check_due(), [])                       # too early
        later = start - timedelta(minutes=29)
        notices = service.check_due(later)
        self.assertEqual(len(notices), 1)
        self.assertIn("Call mom", notices[0].message)
        self.assertEqual(service.check_due(later), [])                  # never twice
        self.assertIsNotNone(task.id)

    def test_completed_task_does_not_notify(self):
        start = datetime.now() + timedelta(hours=2)
        task = self.scheduler.add_task(
            Task("Done already", start.date(), start.time().replace(second=0, microsecond=0)),
            reminder_minutes=0)
        self.scheduler.set_task_status(task.id, "Completed")
        self.assertEqual(ReminderService(self.scheduler).check_due(start + timedelta(minutes=1)), [])

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromModule(__import__("__main__"))
result = unittest.TextTestRunner(verbosity=1).run(suite)

## 10. Cleanup

In [ ]:
db.close()